# PoreGen VAE — Evaluation Results Notebook

This notebook loads the outputs from a completed `train_vae eval` run and
produces an interactive summary with all quantitative metrics, slice comparisons,
uncertainty maps, latent-space audit, and structural metric plots.

**How to use**
1. Set `EVAL_DIR` in cell 2 to your eval output directory, e.g.
   `runs/vae/r03-run-0002-.../eval/20260415-120000-best_checkpoint/`
2. Run all cells (`Kernel > Restart & Run All`).

All outputs are read from disk — no model inference happens here.

In [ ]:
from pathlib import Path
import json, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tifffile
from IPython.display import Image, display, Markdown

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ── Configure this path ───────────────────────────────────────────────────
EVAL_DIR = Path(
    "../runs/vae/r03-run-0002-20260415-140146-archv2-conv_noattn-z16-c32-bs128-lr2e-04-b0.050-fb0.25-klw0-schednone"
    "/eval"
).glob("*/").__next__()   # grab the first (most recent) eval sub-directory

print(f"Eval directory: {EVAL_DIR}")
assert EVAL_DIR.exists(), f"Directory not found: {EVAL_DIR}"

## 1  Scalar metrics summary

In [ ]:
with open(EVAL_DIR / "eval_metrics.json") as f:
    em = json.load(f)

print(f"Run       : {em['run_name']}")
print(f"Checkpoint: {em['checkpoint_path']}")
print(f"Step      : {em['step']:,}")
print(f"Eval tier : {em['eval_config']['tier']}")
print()

In [ ]:
pm = em["patch_metrics"]

def _fmt(d):
    if isinstance(d, dict) and 'mean' in d:
        return f"{d['mean']:.4f} ± {d['std']:.4f}"
    return str(d)

print("=== Patch-level metrics (full test set) ===")
print(f"  n_patches      : {pm['n_patches']:,}")
for key in ['psnr','ssim','mae','dice','precision','recall','f1',
            'porosity_mae','porosity_bias','porosity_volume_mae',
            'sharpness_ratio']:
    print(f"  {key:<22}: {_fmt(pm.get(key, 'n/a'))}")
print(f"  {'recon_std_mean':<22}: {pm.get('recon_std_mean', float('nan')):.6f}  (< 0.01 for LDM)")

In [ ]:
la = em["latent_audit"]
print("=== Latent audit ===")
print(f"  active channels: {la['n_active_channels']} / {la['n_total_channels']}")
print(f"  LDM ready      : {la['ldm_ready']}")
print(f"  flagged low    : channels {la['channels_flagged_low']}")
print(f"  flagged high   : channels {la['channels_flagged_high']}")
print(f"  channels_in_range: {la['channels_in_range']}")

In [ ]:
print("=== Volume-level metrics ===")
for vm in em["volume_metrics"]:
    print(f"\n  Volume: {vm['volume_id']}")
    print(f"    Stoch-mean  PSNR={vm['xct_psnr_stoch']:.2f}  SSIM={vm['xct_ssim_stoch']:.4f}  MAE={vm['xct_mae_stoch']:.4f}")
    print(f"    Single      PSNR={vm['xct_psnr_single']:.2f}  SSIM={vm['xct_ssim_single']:.4f}  MAE={vm['xct_mae_single']:.4f}")
    print(f"    Mu (determ) PSNR={vm['xct_psnr_mu']:.2f}  SSIM={vm['xct_ssim_mu']:.4f}  MAE={vm['xct_mae_mu']:.4f}")
    print(f"    Dice={vm['dice']:.4f}  IoU={vm['iou']:.4f}  por_mae={vm['porosity_mae']:.4f}")
    print(f"    Pore count GT={vm['pore_count_gt']}  pred={vm['pore_count_pred']}")
    if vm.get('s2_wasserstein') is not None:
        print(f"    S2 W1={vm['s2_wasserstein']:.4f}")
    if vm.get('psd_wasserstein') is not None:
        print(f"    PSD W1={vm['psd_wasserstein']:.4f}")
    print(f"    Boundary MAE XCT={vm['xct_boundary_mae']:.5f}  mask={vm['mask_boundary_mae']:.5f}")
    print(f"    recon_std_mean XCT={vm['xct_recon_std_mean']:.6f}  mask={vm['mask_recon_std_mean']:.6f}")

## 2  Latent audit plots

In [ ]:
la_png = EVAL_DIR / "latent_audit.png"
if la_png.exists():
    display(Image(str(la_png)))
else:
    print("latent_audit.png not found — regenerating from JSON …")
    from poregen.eval.metrics import LatentAudit
    from poregen.eval.visualise import save_latent_audit_plot
    from poregen.eval.config import EvalConfig
    audit = LatentAudit(**la)
    cfg = EvalConfig.from_dict(em['eval_config'])
    save_latent_audit_plot(audit, EVAL_DIR,
                           ldm_sigma_low=cfg.ldm_sigma_low,
                           ldm_sigma_high=cfg.ldm_sigma_high)
    display(Image(str(la_png)))

## 3  Porosity scatter

In [ ]:
por_png = EVAL_DIR / "porosity_scatter.png"
if por_png.exists():
    display(Image(str(por_png)))
else:
    print("porosity_scatter.png not found — plotting from patch_arrays.npz …")
    patch_npz_path = EVAL_DIR / "patch_arrays.npz"
    if patch_npz_path.exists():
        data = np.load(str(patch_npz_path), allow_pickle=True)
        gt   = data["porosity_gt"]
        pred = data["porosity_pred"]
        fig, ax = plt.subplots(figsize=(5,5))
        ax.scatter(gt, pred, s=2, alpha=0.3, c='steelblue')
        lim = max(gt.max(), pred.max()) * 1.05
        ax.plot([0,lim],[0,lim],'r--',lw=1)
        ax.set_xlabel('φ GT'); ax.set_ylabel('φ Pred')
        ax.set_title('Patch porosity scatter')
        plt.tight_layout(); plt.show()
    else:
        print("patch_arrays.npz also not found.")

## 4  Prior samples

In [ ]:
prior_png = EVAL_DIR / "prior_samples.png"
if prior_png.exists():
    display(Image(str(prior_png)))
else:
    print("prior_samples.png not found.")

## 5  Volume reconstructions

In [ ]:
vol_dirs = sorted((EVAL_DIR / "volumes").iterdir()) if (EVAL_DIR / "volumes").exists() else []
print(f"Found {len(vol_dirs)} volume directories:")
for vd in vol_dirs:
    print(f"  {vd.name}")

In [ ]:
def show_volume_slices(vol_dir: Path, mode: str = "stoch_mean", axis: str = "axial"):
    """Display a slice comparison PNG for one volume."""
    vol_name = vol_dir.name
    png = vol_dir / f"{vol_name}_slices_{mode}_{axis}.png"
    if png.exists():
        display(Markdown(f"### {vol_name} — {mode} — {axis}"))
        display(Image(str(png)))
    else:
        print(f"Not found: {png}")

# Show stoch_mean axial slices for every volume
for vd in vol_dirs:
    show_volume_slices(vd, mode="stoch_mean", axis="axial")

In [ ]:
def show_all_modes_for_volume(vol_dir: Path, axis: str = "axial"):
    """Side-by-side slice PNGs for all three reconstruction modes."""
    vol_name = vol_dir.name
    display(Markdown(f"### {vol_name} — all modes — {axis}"))
    for mode in ("stoch_mean", "stoch_single", "mu"):
        png = vol_dir / f"{vol_name}_slices_{mode}_{axis}.png"
        if png.exists():
            display(Markdown(f"**{mode}**"))
            display(Image(str(png)))
        else:
            print(f"Missing: {png.name}")

# Show all three modes for the first volume
if vol_dirs:
    show_all_modes_for_volume(vol_dirs[0], axis="axial")

## 6  Uncertainty maps (voxel-wise std)

In [ ]:
for vd in vol_dirs:
    vol_name = vd.name
    for kind, axis in [("xct","axial"),("mask","axial")]:
        png = vd / f"{vol_name}_std_{kind}_{axis}.png"
        if png.exists():
            display(Markdown(f"**{vol_name}  {kind} std  {axis}**"))
            display(Image(str(png)))

## 7  S₂(r) and PSD plots

In [ ]:
for vd in vol_dirs:
    vol_name = vd.name
    for suffix in ("_s2r.png", "_psd.png"):
        png = vd / (vol_name + suffix)
        if png.exists():
            display(Markdown(f"**{vol_name}  {suffix.strip('_.png')}**"))
            display(Image(str(png)))

## 8  Load NPZ arrays for custom analysis

In [ ]:
if vol_dirs:
    vd = vol_dirs[0]
    vol_name = vd.name
    npz_path = vd / f"{vol_name}_arrays.npz"
    if npz_path.exists():
        data = np.load(str(npz_path))
        print("Arrays in NPZ:", list(data.files))
        print(f"  xct_gt shape: {data['xct_gt'].shape}  dtype: {data['xct_gt'].dtype}")
        print(f"  xct_stoch_mean range: [{data['xct_stoch_mean'].min():.3f}, {data['xct_stoch_mean'].max():.3f}]")
        print(f"  xct_stoch_std max: {data['xct_stoch_std'].max():.5f}  mean: {data['xct_stoch_std'].mean():.6f}")

In [ ]:
# Custom mid-axial comparison: GT vs stoch_mean vs mu
if vol_dirs:
    vd = vol_dirs[0]
    vol_name = vd.name
    npz_path = vd / f"{vol_name}_arrays.npz"
    if npz_path.exists():
        data = np.load(str(npz_path))
        D = data['xct_gt'].shape[0]
        mid = D // 2

        fig, axes = plt.subplots(2, 4, figsize=(18, 8))
        fig.suptitle(f"{vol_name}  —  mid-axial slice (z={mid})", fontsize=11)

        panels_top = [
            ("GT XCT",          data['xct_gt'][mid],           "gray"),
            ("Stoch-mean XCT",  data['xct_stoch_mean'][mid],   "gray"),
            ("Single pass XCT", data['xct_stoch_single'][mid],  "gray"),
            ("Mu (determ) XCT", data['xct_mu'][mid],           "gray"),
        ]
        panels_bot = [
            ("GT mask",          data['mask_gt'][mid],           "binary_r"),
            ("Stoch-mean mask",  data['mask_stoch_mean'][mid],   "RdBu_r"),
            ("XCT std",          data['xct_stoch_std'][mid],     "plasma"),
            ("Mask std",         data['mask_stoch_std'][mid],    "plasma"),
        ]

        for col, (title, img, cmap) in enumerate(panels_top):
            axes[0][col].imshow(img, cmap=cmap, vmin=0, vmax=1)
            axes[0][col].set_title(title, fontsize=9)
            axes[0][col].axis('off')

        for col, (title, img, cmap) in enumerate(panels_bot):
            vmax = img.max() if cmap == 'plasma' else 1
            axes[1][col].imshow(img, cmap=cmap, vmin=0, vmax=vmax)
            axes[1][col].set_title(title, fontsize=9)
            axes[1][col].axis('off')

        plt.tight_layout()
        plt.show()

## 9  Porosity scatter from raw arrays

In [ ]:
patch_npz = EVAL_DIR / "patch_arrays.npz"
if patch_npz.exists():
    d = np.load(str(patch_npz), allow_pickle=True)
    gt   = d["porosity_gt"]
    pred = d["porosity_pred"]
    vids = d["volume_ids"]

    unique_vids = sorted(set(vids))
    cmap = plt.get_cmap("tab20", len(unique_vids))
    vid2c = {v: i for i, v in enumerate(unique_vids)}

    fig, ax = plt.subplots(figsize=(6,6))
    sc = ax.scatter(gt, pred,
                    c=[vid2c[v] for v in vids],
                    cmap=cmap, vmin=0, vmax=len(unique_vids),
                    s=3, alpha=0.4)
    lim = max(gt.max(), pred.max()) * 1.05
    ax.plot([0,lim],[0,lim],'k--',lw=1)
    mae = float(np.abs(gt - pred).mean())
    ax.set_xlabel('φ GT'); ax.set_ylabel('φ Pred')
    ax.set_title(f'Patch porosity  MAE={mae:.4f}  n={len(gt):,}')
    cb = plt.colorbar(sc, ax=ax, ticks=np.arange(len(unique_vids))+0.5)
    cb.ax.set_yticklabels([v.replace('MedidasDB__','')[-20:] for v in unique_vids], fontsize=5)
    plt.tight_layout(); plt.show()
else:
    print("patch_arrays.npz not found")

## 10  Per-channel latent statistics

In [ ]:
cfg = em['eval_config']
kl  = np.array(la['kl_per_channel'])
sig = np.array(la['sigma_avg'])
n   = la['n_total_channels']

colors = []
for i in range(n):
    if i in la['channels_flagged_low']:  colors.append('steelblue')
    elif i in la['channels_flagged_high']: colors.append('tomato')
    else: colors.append('seagreen')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(n), kl, color=colors)
axes[0].set_xlabel('Channel'); axes[0].set_ylabel('KL')
axes[0].set_title('Per-channel KL  (blue=collapsed, red=wide)')

axes[1].bar(range(n), sig, color=colors)
axes[1].axhspan(cfg['ldm_sigma_low'], cfg['ldm_sigma_high'],
                color='lightgreen', alpha=0.3,
                label=f"LDM range [{cfg['ldm_sigma_low']}, {cfg['ldm_sigma_high']}]")
axes[1].axhline(cfg['ldm_sigma_low'], color='green', lw=1, ls='--')
axes[1].axhline(cfg['ldm_sigma_high'], color='green', lw=1, ls='--')
axes[1].set_xlabel('Channel'); axes[1].set_ylabel('σ_avg')
ready = 'LDM ready ✓' if la['ldm_ready'] else 'NOT ready ✗'
axes[1].set_title(f'Per-channel σ_avg  {ready}')
axes[1].legend(fontsize=8)

plt.suptitle(f"Latent audit  {la['n_active_channels']}/{n} active  |  recon_std_mean = {pm.get('recon_std_mean', float('nan')):.6f}",
             fontsize=11)
plt.tight_layout()
plt.show()

## 11  Volume-level metric comparison table

In [ ]:
import pandas as pd

rows = []
for vm in em['volume_metrics']:
    short = vm['volume_id'].replace('MedidasDB__','')
    rows.append({
        'Volume': short[:40],
        'Por GT': vm['porosity_gt'],
        'Por MAE': vm['porosity_mae'],
        'PSNR stoch': vm['xct_psnr_stoch'],
        'PSNR mu': vm['xct_psnr_mu'],
        'SSIM stoch': vm['xct_ssim_stoch'],
        'Dice': vm['dice'],
        'S2 W1': vm.get('s2_wasserstein'),
        'PSD W1': vm.get('psd_wasserstein'),
        'Pores GT': vm['pore_count_gt'],
        'Pores pred': vm['pore_count_pred'],
        'Bnd MAE': vm['xct_boundary_mae'],
        'std mean': vm['xct_recon_std_mean'],
    })

df = pd.DataFrame(rows).set_index('Volume')
df.style.format({
    'Por GT': '{:.4f}', 'Por MAE': '{:.4f}',
    'PSNR stoch': '{:.2f}', 'PSNR mu': '{:.2f}',
    'SSIM stoch': '{:.4f}', 'Dice': '{:.4f}',
    'S2 W1': '{:.4f}', 'PSD W1': '{:.4f}',
    'Bnd MAE': '{:.5f}', 'std mean': '{:.6f}',
})

## 12  3-D pore GIFs (paper tier only)

In [ ]:
for vd in vol_dirs:
    vol_name = vd.name
    for size in ('small', 'medium', 'large'):
        gif = vd / f"{vol_name}_pore_{size}.gif"
        if gif.exists():
            display(Markdown(f"**{vol_name}  {size} pore**"))
            display(Image(str(gif)))
print("(GIFs are generated only when run_pore_gifs=True in the eval config)")

## 13  Quick diagnostic: PSNR stoch vs mu

In [ ]:
if len(em['volume_metrics']) >= 2:
    names  = [v['volume_id'].replace('MedidasDB__','')[:30] for v in em['volume_metrics']]
    stoch  = [v['xct_psnr_stoch'] for v in em['volume_metrics']]
    mu_val = [v['xct_psnr_mu']    for v in em['volume_metrics']]

    x = np.arange(len(names))
    w = 0.35
    fig, ax = plt.subplots(figsize=(max(6, len(names)*2), 4))
    ax.bar(x - w/2, stoch,  w, label='Stoch-mean', color='royalblue')
    ax.bar(x + w/2, mu_val, w, label='Mu (determ)', color='tomato')
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right', fontsize=8)
    ax.set_ylabel('PSNR (dB)')
    ax.set_title('PSNR: stochastic mean vs deterministic μ')
    ax.legend()
    plt.tight_layout(); plt.show()
else:
    print('Fewer than 2 volumes — bar chart skipped.')

## 14  Export summary to CSV

In [ ]:
import pandas as pd

# Patch metrics
patch_rows = []
for key in ['psnr','ssim','mae','dice','precision','recall','f1',
            'porosity_mae','porosity_bias','sharpness_ratio']:
    v = pm.get(key, {})
    if isinstance(v, dict):
        patch_rows.append({'metric': key, 'mean': v.get('mean'), 'std': v.get('std')})
    else:
        patch_rows.append({'metric': key, 'mean': v, 'std': None})
patch_rows.append({'metric': 'recon_std_mean', 'mean': pm.get('recon_std_mean'), 'std': None})

df_patch = pd.DataFrame(patch_rows)
out_csv = EVAL_DIR / "patch_metrics_summary.csv"
df_patch.to_csv(str(out_csv), index=False)
print(f"Saved: {out_csv}")
df_patch